# Imports??

In [2]:
import os
import numpy as np
import ee
import geemap
from dotenv import load_dotenv

load_dotenv()
project_id = os.getenv("GOOGLE_CLOUD_PROJECT_ID") # insert your id here

ee.Authenticate(force=False)
ee.Initialize(project=project_id)

# Helper Functions


In [22]:
def mask_clouds_landsat(image:ee.Image):
    """Given an Image from the Landsat 8 Collection 2 Tier 1 dataset, applies a mask that removes 
    clouds and shadows."""

    # Select the Quality Assessment band
    qa = image.select('QA_PIXEL')

    # Bits 3 and 4 are Cloud and Cloud Shadow, respectively.
    # We create a mask where these bits are set to 0 (Clear)
    cloud_bit = 1 << 3
    shadow_bit = 1 << 4

    # Combined mask: pixel must have neither cloud nor shadow
    mask = qa.bitwiseAnd(cloud_bit).eq(0).And(
           qa.bitwiseAnd(shadow_bit).eq(0))

    # Apply the mask to the image (making clouds/shadows transparent)
    return image.updateMask(mask)


def add_coords(feature:ee.Feature):
    """Given a Feature, adds the longitude and latitude as properties."""
    return feature.set({
        'longitude': feature.geometry().coordinates().get(0),
        'latitude': feature.geometry().coordinates().get(1)
    })


def get_bands(year: int, month: int, day: int, location:ee.Geometry) -> ee.ImageCollection:
    """Gets the Landsat 8 Collection 2 Tier 1 image bands from (year, month, day) to (year+1, month, day) for the provided geometry."""
    return ee.ImageCollection("LANDSAT/LC08/C02/T1_L2") \
        .filterDate(ee.Date.fromYMD(year, month, day), ee.Date.fromYMD(year+1, month, day)) \
        .map(mask_clouds_landsat)

def get_indices(img_collection: ee.ImageCollection) -> ee.Image:
    """Given an ImageCollection from the Landsat 8 Collection 2 Tier 1 dataset, creates an image with the median
    Red, Near Infrared, Shortwave Infrared 1 and 2 spectral bands, as well as NDVI, NBR, NDMI indices.
    Learn more about spectral indices here: https://www.geo.university/pages/spectral-indices-in-remote-sensing-and-how-to-interpret-them"""
    # Select original bands
    selected_bands = img_collection.select(['SR_B4', 'SR_B5', 'SR_B6', 'SR_B7']).median()
    # print('Available bands:', img_collection.first().bandNames().getInfo())
    # Calculate indices using normalizedDifference
    ndvi = selected_bands.normalizedDifference(['SR_B5', 'SR_B4']).rename('NDVI') # NIR - Red (SR_B5 - SR_B4)
    nbr = selected_bands.normalizedDifference(['SR_B5', 'SR_B7']).rename('NBR')   # NIR - SWIR2 (SR_B5 - SR_B7)
    ndmi = selected_bands.normalizedDifference(['SR_B5', 'SR_B6']).rename('NDMI') # NIR - SWIR1 (SR_B5 - SR_B6)

    # Return original bands plus calculated indices
    return selected_bands.addBands([ndvi, nbr, ndmi])


def process_yearly_landsat(year:int, month:int, day:int, base_year:int, base_month:int, base_day:int, location:ee.Geometry)->ee.Image:
    """Given year, month, day and base_year, base_month, base_day, gets bands, indices, and year-to-base_year deltas of the Landsat 8 Collection 2 Tier 1 dataset as per get_indices(), from year to base_year for the provided location (a Geometry)"""
    # gets lagged spectral bands, spectral indices, and deltas from year to base_year
    start_date = ee.Date.fromYMD(year, month, day)


    # Process the target year
    img = get_indices(get_bands(year, month, day, location))
    last_year_img = get_indices(get_bands(base_year, base_month, base_day, location))
    if year != base_year:
        print(year, base_year)
        deltas = img.subtract(last_year_img).rename(img.bandNames().map(lambda n: ee.String(n).cat('_delta')))
        img = img.addBands(deltas)


    # Calculate Lags for labeling - Added .toInt() to avoid '.0' in band names
    lag = ee.Number(base_year).subtract(year)
    lag_suffix = ee.String('_lag').cat(lag.toInt().format())

    # Rename year bands
    img_renamed = img.rename(img.bandNames().map(lambda n: ee.String(n).cat(lag_suffix)))

    return img_renamed \
                .set('year', year) \
                .set('lag', lag) \
                .set('base_year', base_year) \
                .set('system:time_start', start_date.millis())

In [23]:
def get_data(base_year:int, base_month, base_day, location:ee.Geometry, filename:str, foldername, numPoints:int)->None:
  """Given a year, a location as a Geometry, a filename, and numPoints, 
     extracts numPoints points from image bands/indices for each target value (whether or not the pixel was deforested in year).
     Then calculates bands and deltas of band up to four years before the base date. Saves to filename."""

  # get targets
  hansen = ee.Image("UMD/hansen/global_forest_change_2025_v1_13")
  lossyear = hansen.select('lossyear')
  datamask = hansen.select('datamask')
  treecover = hansen.select('treecover2000')

  land_forest_mask = datamask.eq(1).And(treecover.gt(50))
  target_band = lossyear.unmask(0).updateMask(land_forest_mask).rename('target')
  valid_loss_mask = target_band.eq(0).Or(target_band.eq(base_year-2000))
  class_band = ee.Image(0).updateMask(land_forest_mask) \
                          .where(target_band.eq(base_year-2000), 1) \
                          .rename('class')
  # ee.Image(0).updateMask(valid_loss_mask).where(target_band.eq(base_year-2000), 1).where(target_band.eq(0), 0).rename('class')

  # sample points to include in the dataset
  sampled_points = class_band.stratifiedSample(
    numPoints=numPoints,      # Number of points to sample for each class
    classBand='class',    # The name of the band containing the class labels
    region=location,      # The geographical region to sample within
    scale=30,             # The nominal scale (resolution) in meters at which to sample
    geometries=True,      # Include feature geometries (points) in the output
    seed=0                # A random seed for reproducibility
  )
  sampled_points = sampled_points.map(add_coords)

  experimental = sampled_points.filter(ee.Filter.eq('class', 1))
  control = sampled_points.filter(ee.Filter.eq('class', 0))

  full_set = experimental.merge(control)

  # get features
  years = np.arange(base_year-4, base_year+1).tolist()
  collection = ee.ImageCollection([process_yearly_landsat(y, base_month, base_day, base_year, base_month, base_day, location) for y in years])
 
  # merge features and target for our points
  features_image = collection.toBands()
  targets_with_features = features_image.sampleRegions(
      collection=full_set,
      properties=['class', 'longitude', 'latitude'], # Include the 'class', 'longitude', and 'latitude' properties
      scale=30,
      tileScale=16
  )

  # export dataset
  export_task = ee.batch.Export.table.toDrive(
    collection=targets_with_features,
    description=f'exporting {filename}',
    folder=foldername,
    fileNamePrefix=filename,
    fileFormat='CSV'
  )
  export_task.start()

# Generating the Data

## Global Model

### Train Sets

In [5]:
# Get three years of training data in the Amazon Rainforest, Siberian Taiga, and Borneo rainforest. Come final model evaluation, I will also extract 2025 as my test set

amazon_roi = ee.Geometry.Rectangle([-75, -15, -50, 5])
siberian_taiga_roi = ee.Geometry.Rectangle([80, 50, 120, 70])
borneo_roi = ee.Geometry.Rectangle([108, -5, 119, 8])


In [ ]:
get_data(2022, 1, 1, amazon_roi, 'amazon_train_2022_year_expanded_class', 'ee_exports_post_pivot', 3000)
get_data(2023, 1, 1, amazon_roi, 'amazon_train_2023_year_expanded_class', 'ee_exports_post_pivot', 3000)
get_data(2024, 1, 1, amazon_roi, 'amazon_train_2024_year_expanded_class', 'ee_exports_post_pivot', 3000)
get_data(2022, 1, 1, siberian_taiga_roi, 'taiga_train_2022_year_expanded_class', 'ee_exports_post_pivot', 3000)
get_data(2023, 1,1, siberian_taiga_roi, 'taiga_train_2023_year_expanded_class', 'ee_exports_post_pivot', 3000)
get_data(2024, 1, 1, siberian_taiga_roi, 'taiga_train_2024_year_expanded_class', 'ee_exports_post_pivot', 3000)
get_data(2022, 1, 1, borneo_roi, 'borneo_train_2022_year_expanded_class', 'ee_exports_post_pivot', 3000)
get_data(2023, 1, 1, borneo_roi, 'borneo_train_2023_year_expanded_class', 'ee_exports_post_pivot', 3000)
get_data(2024, 1, 1, borneo_roi, 'borneo_train_2024_year_expanded_class', 'ee_exports_post_pivot', 3000)

### Test Sets

In [ ]:
chaco_roi = ee.Geometry.Rectangle([-60.5, -23.0, -59.5, -22.0])
se_usa_roi = ee.Geometry.Rectangle([-84.0, 31.0, -83.0, 32.0])
andes_roi = ee.Geometry.Rectangle([-78.0, -1.0, -77.0, 0.0])

In [ ]:
get_data(2025, 1, 1, amazon_roi, 'amazon_test_2025', 'ee_exports_post_pivot', 3000)
get_data(2025, 1, 1, siberian_taiga_roi, 'taiga_test_2025', 'ee_exports_post_pivot', 3000)
get_data(2025,1, 1, borneo_roi, 'borneo_test_2025', 'ee_exports_post_pivot', 3000)

In [ ]:
get_data(2025, 1, 1, chaco_roi, 'gran_chaco_test_2025', 'ee_exports_post_pivot', 3000)
get_data(2025, 1, 1, se_usa_roi, 'se_usa_test_2025', 'ee_exports_post_pivot', 3000)
get_data(2025,1, 1, andes_roi, 'andes_test_2025', 'ee_exports_post_pivot', 3000)

2021 2025
2022 2025
2023 2025
2024 2025
2021 2025
2022 2025
2023 2025
2024 2025
2021 2025
2022 2025
2023 2025
2024 2025


## Per-Type Models

In [24]:
# --- TROPICAL TRAIN SET ---

# 1. Amazon Basin (Brazil - Amazonas)
# Driver: Large-scale cattle ranching and massive soy farm clear-cuts.
# Physics: Flat topography, extremely dense continuous evergreen canopy.
tropical_train_amazon = ee.Geometry.Rectangle([-65.0, -5.0, -63.0, -3.0])

# 2. Congo Basin (Democratic Republic of the Congo - Tshuapa)
# Driver: Small-holder shifting agriculture and artisanal logging.
# Physics: More fragmented canopy than the Amazon, highly persistent cloud cover.
tropical_train_congo = ee.Geometry.Rectangle([21.0, -1.0, 23.0, 1.0])

# 3. Southeast Asia (Indonesia - Central Kalimantan)
# Driver: Industrial palm oil plantations.
# Physics: Peat swamp forests. Deforestation here often involves massive drainage 
# canals and controlled burns, creating unique spectral burn scars.
tropical_train_borneo = ee.Geometry.Rectangle([113.0, 1.0, 115.0, 3.0])

# --- TROPICAL TEST SET ---

# 4. Australasia (Papua New Guinea - East Sepik)
# Driver: Rapidly accelerating foreign commercial logging in highly intact areas.
# Physics: Severe mountainous topography. If your model survives the topographic 
# shadows here without throwing false positives, it is bulletproof.
tropical_test_png = ee.Geometry.Rectangle([142.0, -7.0, 144.0, -5.0])

In [25]:
# --- TEMPERATE TRAIN SET ---

# 1. North American Broadleaf (USA - Appalachian Mountains)
# Driver: Mixed selective logging and urban/suburban expansion.
# Physics: Classic deciduous forest. High seasonal variance. Topographically complex.
temperate_train_appalachia = ee.Geometry.Rectangle([-80.0, 38.0, -78.0, 40.0])

# 2. European Mixed Forest (Romania - Carpathian Mountains)
# Driver: Some of the most intense illegal logging of old-growth forests in Europe.
# Physics: A chaotic mix of broadleaf and coniferous trees on steep terrain.
temperate_train_carpathians = ee.Geometry.Rectangle([24.0, 46.0, 26.0, 47.0])

# 3. East Asian Temperate (Russia - Primorsky Krai)
# Driver: Legal and illegal timber extraction for export.
# Physics: Highly unique biome where boreal species mix with subtropical species. 
# Snow cover in winter introduces extreme albedo spikes.
temperate_train_fareast = ee.Geometry.Rectangle([134.0, 45.0, 136.0, 47.0])

# --- TEMPERATE TEST SET ---

# 4. Southern Hemisphere (Chile - Valdivian Temperate Rainforest)
# Driver: Conversion of native ancient forests to commercial eucalyptus plantations.
# Physics: Extremely wet, evergreen temperate rainforest. Completely different 
# species composition and seasonal timing than the Northern Hemisphere.
temperate_test_valdivian = ee.Geometry.Rectangle([-73.0, -42.0, -72.0, -40.0])

In [26]:
# --- BOREAL TRAIN SET ---

# 1. North American Boreal (Canada - Central Quebec)
# Driver: Massive industrial timber and pulp extraction.
# Physics: Millions of lakes disrupt the landscape. Canopy is naturally sparse, 
# meaning the satellite sees a lot of the understory and snowpack.
boreal_train_quebec = ee.Geometry.Rectangle([-74.0, 49.0, -72.0, 51.0])

# 2. Eurasian Boreal (Russia - Krasnoyarsk Krai)
# Driver: Catastrophic, massive-scale wildfires and remote logging.
# Physics: The largest unbroken forest on Earth. Extreme temperature swings 
# (-40°C winter to +30°C summer) dictate the spectral baselines.
boreal_train_siberia = ee.Geometry.Rectangle([90.0, 60.0, 93.0, 62.0])

# 3. Fennoscandia (Central Sweden)
# Driver: Highly regulated, intense commercial clear-cutting.
# Physics: Almost no "primary" forest remains; it is essentially a massive, 
# well-oiled tree farm. Perfect for learning the geometry of legal clear-cuts.
boreal_train_sweden = ee.Geometry.Rectangle([14.0, 61.0, 16.0, 63.0])

# --- BOREAL TEST SET ---

# 4. High-Latitude Boreal (USA - Interior Alaska)
# Driver: Climate-driven permafrost thaw, wildfires, and localized clearing.
# Physics: Extreme sun angles. In the winter, the sun barely clears the horizon, 
# creating massive, elongated tree shadows that drastically alter reflectance.
boreal_test_alaska = ee.Geometry.Rectangle([-150.0, 64.0, -147.0, 66.0])

In [27]:
for train_year in range(2022, 2025):
    # Tropical
    get_data(train_year, 1, 1, tropical_train_amazon, f'amazon_train_{train_year}', 'tropical', 3000)
    get_data(train_year, 1, 1, tropical_train_congo, f'congo_train_{train_year}', 'tropical', 3000)
    get_data(train_year,1, 1, tropical_train_borneo, f'borneo_train_{train_year}', 'tropical', 3000)
    # Temperate
    get_data(train_year,1, 1, temperate_train_appalachia, f'appalachia_train_{train_year}', 'temperate', 3000)
    get_data(train_year,1, 1, temperate_train_carpathians, f'carpathians_train_{train_year}', 'temperate', 3000)
    get_data(train_year,1, 1, temperate_train_fareast, f'fareast_train_{train_year}', 'temperate', 3000)

    # Boreal
    get_data(train_year,1, 1, boreal_train_quebec, f'quebec_train_{train_year}', 'boreal', 3000)
    get_data(train_year,1, 1, boreal_train_siberia, f'siberia_train_{train_year}', 'boreal', 3000)
    get_data(train_year,1, 1, boreal_train_sweden, f'sweden_train_{train_year}', 'boreal', 3000)

2018 2022
2019 2022
2020 2022
2021 2022
2018 2022
2019 2022
2020 2022
2021 2022
2018 2022
2019 2022
2020 2022
2021 2022
2018 2022
2019 2022
2020 2022
2021 2022
2018 2022
2019 2022
2020 2022
2021 2022
2018 2022
2019 2022
2020 2022
2021 2022
2018 2022
2019 2022
2020 2022
2021 2022
2018 2022
2019 2022
2020 2022
2021 2022
2018 2022
2019 2022
2020 2022
2021 2022
2019 2023
2020 2023
2021 2023
2022 2023
2019 2023
2020 2023
2021 2023
2022 2023
2019 2023
2020 2023
2021 2023
2022 2023
2019 2023
2020 2023
2021 2023
2022 2023
2019 2023
2020 2023
2021 2023
2022 2023
2019 2023
2020 2023
2021 2023
2022 2023
2019 2023
2020 2023
2021 2023
2022 2023
2019 2023
2020 2023
2021 2023
2022 2023
2019 2023
2020 2023
2021 2023
2022 2023
2020 2024
2021 2024
2022 2024
2023 2024
2020 2024
2021 2024
2022 2024
2023 2024
2020 2024
2021 2024
2022 2024
2023 2024
2020 2024
2021 2024
2022 2024
2023 2024
2020 2024
2021 2024
2022 2024
2023 2024
2020 2024
2021 2024
2022 2024
2023 2024
2020 2024
2021 2024
2022 2024
2023 2024


In [ ]:
# Tropical
get_data(2025, 1, 1, tropical_train_amazon, f'amazon_test_2025', 'tropical', 3000)
get_data(2025, 1, 1, tropical_train_congo, f'congo_test_2025', 'tropical', 3000)
get_data(2025,1, 1, tropical_train_borneo, f'borneo_test_2025', 'tropical', 3000)
get_data(2025,1, 1, tropical_test_png, f'png_test_2025', 'tropical', 3000)

# Temperate
get_data(2025,1, 1, temperate_train_appalachia, f'appalachia_test_2025', 'temperate', 3000)
get_data(2025,1, 1, temperate_train_carpathians, f'carpathians_test_2025', 'temperate', 3000)
get_data(2025,1, 1, temperate_train_fareast, f'fareast_test_2025', 'temperate', 3000)
get_data(2025,1, 1, temperate_test_valdivian, f'valdivian_test_2025', 'temperate', 3000)

# Boreal
get_data(2025,1, 1, boreal_train_quebec, f'quebec_test_2025', 'boreal', 3000)
get_data(2025,1, 1, boreal_train_siberia, f'siberia_test_2025', 'boreal', 3000)
get_data(2025,1, 1, boreal_train_sweden, f'sweden_test_2025', 'boreal', 3000)
get_data(2025,1, 1, boreal_test_alaska, f'alaska_test_2025', 'boreal', 3000)

2021 2025
2022 2025
2023 2025
2024 2025
2021 2025
2022 2025
2023 2025
2024 2025
2021 2025
2022 2025
2023 2025
2024 2025
2021 2025
2022 2025
2023 2025
2024 2025
2021 2025
2022 2025
2023 2025
2024 2025
2021 2025
2022 2025
2023 2025
2024 2025
2021 2025
2022 2025
2023 2025
2024 2025
2021 2025
2022 2025
2023 2025
2024 2025
2021 2025
2022 2025
2023 2025
2024 2025
2021 2025
2022 2025
2023 2025
2024 2025
2021 2025
2022 2025
2023 2025
2024 2025
2021 2025
2022 2025
2023 2025
2024 2025


# Map

This is a cool visualization, comparing the landsat red band (high values indicate forests) with lossyear from the Hansen dataset.

In [14]:
hansen = ee.Image("UMD/hansen/global_forest_change_2024_v1_12")

# 3. Create an Interactive Map
Map = geemap.Map()

# 4. Define Visualization Parameters
# 'lossyear' is encoded 1-24; we'll visualize it with a gradient
loss_viz = {
    'bands': ['lossyear'],
    'min': 1,
    'max': 24,
    'palette': ['yellow', 'orange', 'red']
}

# Visualization for first_b30 (red band from 2000) to detect forests
# Green for high values (forest), black for low values
first_b30_forest_viz = {
    'min': 0,
    'max': 255,
    'palette': ['black', 'yellow', 'green']
}

# 5. Add layers to the map
# treecover2000 as a background (grayscale)
Map.addLayer(hansen.select('treecover2000'), {'max': 100}, 'Tree Cover 2000')

# Highlight pixels where loss occurred, colored by year
loss_mask = hansen.select('lossyear').gt(0).updateMask(hansen.select('datamask').eq(1))
Map.addLayer(hansen.select('loss'), {'min':0, 'max':1}, name='Loss')
Map.addLayer(hansen.select('first_b30'), first_b30_forest_viz, 'Landsat Red 2000 (Forest)')
Map.addLayer(hansen.select('lossyear'), loss_viz, 'Year of Forest Loss')

tropical_params = {
    'color': 'red',       # Border color
    'fillColor': 'blue',  # Fill color (hex codes or named colors work)
    'width': 3            # Stroke width
}

temperate_params = {
    'color': 'green',       # Border color
    'fillColor': 'blue',  # Fill color (hex codes or named colors work)
    'width': 3            # Stroke width
}

boreal_params = {
    'color': 'blue',       # Border color
    'fillColor': 'blue',  # Fill color (hex codes or named colors work)
    'width': 3            # Stroke width
}

# regions of sampling
Map.addLayer(tropical_train_amazon, tropical_params, 'Amazon')
Map.addLayer(tropical_train_congo, tropical_params, 'Congo')
Map.addLayer(tropical_train_borneo, tropical_params, 'Borneo')
Map.addLayer(tropical_test_png, tropical_params, 'PNG')

Map.addLayer(temperate_train_appalachia, temperate_params, 'Appalachia')
Map.addLayer(temperate_train_carpathians, temperate_params, 'Carpathians')
Map.addLayer(temperate_train_fareast, temperate_params, 'Far East')
Map.addLayer(temperate_test_valdivian, temperate_params, 'Valdivians')

Map.addLayer(boreal_train_quebec, boreal_params, 'Quebec')
Map.addLayer(boreal_train_siberia, boreal_params, 'Siberia')
Map.addLayer(boreal_train_sweden, boreal_params, 'Sweden')
Map.addLayer(boreal_test_alaska, boreal_params, 'Alaska')

# 6. Center the map (Longitude, Latitude, Zoom Level)
Map.setCenter(-62.3, -9.2, 10) # Rondônia, Brazil (High deforestation)

Map

Map(center=[-9.2, -62.3], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topright',…